# BPR-MF and LightGCN Baselines

Notebook nay tai su dung preprocessing tu `main.ipynb`, tu viet `BPRMF`, dung `PyG LightGCN`, va huan luyen bang mot training loop BPR chung cho ca hai model.

In [3]:
import json
import math
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.datasets.movie_lens_1m import MovieLens1M
from torch_geometric.nn.models import LightGCN
from tqdm.auto import tqdm

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f'Torch: {torch.__version__}')
print(f'PyG:   {LightGCN.__module__.split(".")[0]}')

Torch: 2.13.0+cpu
PyG:   torch_geometric


In [4]:
BASE_CONFIG = {
    'dataset_root': './data/MovieLens1M',
    'artifacts_root': './artifacts',
    'positive_threshold': 4.0,
    'min_interactions': 5,
    'embedding_dim': 64,
    'num_layers': 3,
    'batch_size': 2048,
    'epochs': 50,
    'lr': 1e-3,
    'weight_decay': 0.0,
    'reg_weight': 1e-4,
    'patience': 10,
    'monitor': 'recall@20',
    'top_k': [10, 20],
    'eval_batch_users': 256,
    'seeds': [42, 52, 62],
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'save_curves': True,
}

BASE_CONFIG

{'dataset_root': './data/MovieLens1M',
 'artifacts_root': './artifacts',
 'positive_threshold': 4.0,
 'min_interactions': 5,
 'embedding_dim': 64,
 'num_layers': 3,
 'batch_size': 2048,
 'epochs': 50,
 'lr': 0.001,
 'weight_decay': 0.0,
 'reg_weight': 0.0001,
 'patience': 10,
 'monitor': 'recall@20',
 'top_k': [10, 20],
 'eval_batch_users': 256,
 'seeds': [42, 52, 62],
 'device': 'cpu',
 'save_curves': True}

## Preprocessing

In [5]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_movielens_implicit(config: dict) -> dict:
    dataset = MovieLens1M(root=config['dataset_root'])
    data = dataset[0]

    edge_index = data['user', 'rates', 'movie'].edge_index
    ratings = data['user', 'rates', 'movie'].rating.float()
    timestamps = data['user', 'rates', 'movie'].time.long()

    df = pd.DataFrame({
        'user': edge_index[0].cpu().numpy(),
        'movie': edge_index[1].cpu().numpy(),
        'rating': ratings.cpu().numpy(),
        'timestamp': timestamps.cpu().numpy(),
    })

    df = df[df['rating'] >= config['positive_threshold']].copy()

    user_counts = df.groupby('user').size()
    valid_users = user_counts[user_counts >= config['min_interactions']].index
    df = df[df['user'].isin(valid_users)].copy()

    unique_users = np.sort(df['user'].unique())
    unique_movies = np.sort(df['movie'].unique())

    user2idx = {user_id: idx for idx, user_id in enumerate(unique_users)}
    movie2idx = {movie_id: idx for idx, movie_id in enumerate(unique_movies)}

    df['user_idx'] = df['user'].map(user2idx)
    df['movie_idx'] = df['movie'].map(movie2idx)

    num_users = len(unique_users)
    num_items = len(unique_movies)

    df = df.sort_values(['user_idx', 'timestamp']).copy()
    df['split'] = 'train'

    tail_two = df.groupby('user_idx', group_keys=False).tail(2).copy()
    val_idx = tail_two.groupby('user_idx', group_keys=False).head(1).index
    test_idx = tail_two.groupby('user_idx', group_keys=False).tail(1).index

    df.loc[val_idx, 'split'] = 'val'
    df.loc[test_idx, 'split'] = 'test'

    train_df = df[df['split'] == 'train'].copy()
    val_df = df[df['split'] == 'val'].copy()
    test_df = df[df['split'] == 'test'].copy()

    train_edges = torch.tensor(train_df[['user_idx', 'movie_idx']].values, dtype=torch.long)
    val_edges = torch.tensor(val_df[['user_idx', 'movie_idx']].values, dtype=torch.long)
    test_edges = torch.tensor(test_df[['user_idx', 'movie_idx']].values, dtype=torch.long)

    train_user_pos_items = [set() for _ in range(num_users)]
    for user_id, item_id in train_edges.tolist():
        train_user_pos_items[user_id].add(item_id)

    val_items = torch.full((num_users,), -1, dtype=torch.long)
    val_items[val_edges[:, 0]] = val_edges[:, 1]

    test_items = torch.full((num_users,), -1, dtype=torch.long)
    test_items[test_edges[:, 0]] = test_edges[:, 1]

    movie_offset = num_users
    user_nodes = train_edges[:, 0]
    movie_nodes = train_edges[:, 1] + movie_offset
    src = torch.cat([user_nodes, movie_nodes], dim=0)
    dst = torch.cat([movie_nodes, user_nodes], dim=0)
    edge_index_train = torch.stack([src, dst], dim=0)

    stats = {
        'num_users': num_users,
        'num_items': num_items,
        'positive_interactions': int(len(df)),
        'train_interactions': int(len(train_df)),
        'val_interactions': int(len(val_df)),
        'test_interactions': int(len(test_df)),
    }

    return {
        'train_edges': train_edges,
        'val_edges': val_edges,
        'test_edges': test_edges,
        'train_user_pos_items': train_user_pos_items,
        'val_items': val_items,
        'test_items': test_items,
        'edge_index_train': edge_index_train,
        'num_users': num_users,
        'num_items': num_items,
        'stats': stats,
    }


dataset_bundle = load_movielens_implicit(BASE_CONFIG)
dataset_bundle['stats']

{'num_users': 6034,
 'num_items': 3533,
 'positive_interactions': 575272,
 'train_interactions': 563204,
 'val_interactions': 6034,
 'test_interactions': 6034}

## Models and BPR Loss

In [6]:
class BPRMF(nn.Module):
    def __init__(self, num_users: int, num_items: int, embedding_dim: int) -> None:
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def get_user_item_embeddings(self, edge_index: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
        del edge_index
        return self.user_embedding.weight, self.item_embedding.weight

    def bpr_forward(
        self,
        users: torch.Tensor,
        pos_items: torch.Tensor,
        neg_items: torch.Tensor,
        edge_index: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        del edge_index
        user_vec = self.user_embedding(users)
        pos_vec = self.item_embedding(pos_items)
        neg_vec = self.item_embedding(neg_items)
        pos_scores = (user_vec * pos_vec).sum(dim=-1)
        neg_scores = (user_vec * neg_vec).sum(dim=-1)
        reg = (user_vec.pow(2).sum() + pos_vec.pow(2).sum() + neg_vec.pow(2).sum()) / users.size(0)
        return pos_scores, neg_scores, reg


class LightGCNBaseline(nn.Module):
    def __init__(self, num_users: int, num_items: int, embedding_dim: int, num_layers: int) -> None:
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.model = LightGCN(
            num_nodes=num_users + num_items,
            embedding_dim=embedding_dim,
            num_layers=num_layers,
        )

    def get_user_item_embeddings(self, edge_index: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        node_embeddings = self.model.get_embedding(edge_index)
        return node_embeddings[:self.num_users], node_embeddings[self.num_users:]

    def bpr_forward(
        self,
        users: torch.Tensor,
        pos_items: torch.Tensor,
        neg_items: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        user_emb, item_emb = self.get_user_item_embeddings(edge_index)
        user_vec = user_emb[users]
        pos_vec = item_emb[pos_items]
        neg_vec = item_emb[neg_items]
        pos_scores = (user_vec * pos_vec).sum(dim=-1)
        neg_scores = (user_vec * neg_vec).sum(dim=-1)
        reg = (user_vec.pow(2).sum() + pos_vec.pow(2).sum() + neg_vec.pow(2).sum()) / users.size(0)
        return pos_scores, neg_scores, reg


def bpr_loss(
    pos_scores: torch.Tensor,
    neg_scores: torch.Tensor,
    reg_term: torch.Tensor,
    reg_weight: float,
) -> torch.Tensor:
    ranking_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-12).mean()
    return ranking_loss + reg_weight * reg_term


def build_model(model_name: str, data_bundle: dict, config: dict) -> nn.Module:
    if model_name == 'bprmf':
        return BPRMF(
            num_users=data_bundle['num_users'],
            num_items=data_bundle['num_items'],
            embedding_dim=config['embedding_dim'],
        )
    if model_name == 'lightgcn':
        return LightGCNBaseline(
            num_users=data_bundle['num_users'],
            num_items=data_bundle['num_items'],
            embedding_dim=config['embedding_dim'],
            num_layers=config['num_layers'],
        )
    raise ValueError(f'Unsupported model: {model_name}')

## Negative Sampling and Evaluation

In [7]:
def sample_negative_items(
    users: torch.Tensor,
    train_user_pos_items: list[set[int]],
    num_items: int,
    device: torch.device,
) -> torch.Tensor:
    negatives = torch.randint(0, num_items, (users.size(0),), device=device)
    user_list = users.detach().cpu().tolist()

    invalid_mask = torch.tensor(
        [negatives[idx].item() in train_user_pos_items[user_id] for idx, user_id in enumerate(user_list)],
        dtype=torch.bool,
        device=device,
    )

    while invalid_mask.any():
        invalid_idx = invalid_mask.nonzero(as_tuple=False).view(-1)
        resampled = torch.randint(0, num_items, (invalid_idx.numel(),), device=device)
        negatives[invalid_idx] = resampled
        invalid_mask = torch.tensor(
            [negatives[idx].item() in train_user_pos_items[user_id] for idx, user_id in enumerate(user_list)],
            dtype=torch.bool,
            device=device,
        )

    return negatives


def metric_at_k(topk_items: torch.Tensor, targets: torch.Tensor, k: int) -> dict[str, float]:
    topk_slice = topk_items[:, :k]
    hits = topk_slice.eq(targets.unsqueeze(1))
    hit_any = hits.any(dim=1)
    ranks = hits.float().argmax(dim=1)
    ndcg = torch.where(
        hit_any,
        1.0 / torch.log2(ranks.float() + 2.0),
        torch.zeros_like(ranks, dtype=torch.float32),
    )
    hit_rate = hit_any.float().mean().item()
    return {
        f'recall@{k}': hit_rate,
        f'hit_rate@{k}': hit_rate,
        f'ndcg@{k}': ndcg.mean().item(),
    }


@torch.no_grad()
def evaluate_model(model: nn.Module, data_bundle: dict, config: dict, split: str) -> dict[str, float]:
    model.eval()
    device = torch.device(config['device'])
    edge_index = data_bundle['edge_index_train'].to(device)
    user_emb, item_emb = model.get_user_item_embeddings(edge_index)

    if split == 'val':
        target_items = data_bundle['val_items']
        extra_exclude = None
    elif split == 'test':
        target_items = data_bundle['test_items']
        extra_exclude = data_bundle['val_items']
    else:
        raise ValueError(f'Unsupported split: {split}')

    valid_users = torch.arange(data_bundle['num_users'])[target_items >= 0]
    max_k = max(config['top_k'])

    aggregates = {f'recall@{k}': 0.0 for k in config['top_k']}
    aggregates.update({f'hit_rate@{k}': 0.0 for k in config['top_k']})
    aggregates.update({f'ndcg@{k}': 0.0 for k in config['top_k']})

    processed = 0
    item_matrix = item_emb.transpose(0, 1)

    for start in range(0, valid_users.numel(), config['eval_batch_users']):
        users = valid_users[start:start + config['eval_batch_users']]
        users_device = users.to(device)
        scores = user_emb[users_device] @ item_matrix

        users_list = users.tolist()
        for row_idx, user_id in enumerate(users_list):
            seen_items = list(data_bundle['train_user_pos_items'][user_id])
            if seen_items:
                scores[row_idx, torch.tensor(seen_items, device=device)] = -torch.inf

            if extra_exclude is not None:
                extra_item = extra_exclude[user_id].item()
                target_item = target_items[user_id].item()
                if extra_item >= 0 and extra_item != target_item:
                    scores[row_idx, extra_item] = -torch.inf

        targets = target_items[users].to(device)
        topk_items = torch.topk(scores, k=max_k, dim=1).indices

        for k in config['top_k']:
            chunk_metrics = metric_at_k(topk_items, targets, k)
            batch_size = users.numel()
            for key, value in chunk_metrics.items():
                aggregates[key] += value * batch_size

        processed += users.numel()

    return {key: value / processed for key, value in aggregates.items()}

## Training

In [8]:
class EarlyStopping:
    def __init__(self, patience: int) -> None:
        self.patience = patience
        self.best_value = -float('inf')
        self.best_epoch = -1
        self.num_bad_epochs = 0

    def step(self, value: float, epoch: int) -> bool:
        if value > self.best_value:
            self.best_value = value
            self.best_epoch = epoch
            self.num_bad_epochs = 0
            return True

        self.num_bad_epochs += 1
        return False

    @property
    def should_stop(self) -> bool:
        return self.num_bad_epochs >= self.patience


def plot_history(history_df: pd.DataFrame, run_dir: Path) -> None:
    if plt is None or history_df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'])
    axes[0].set_title('Train BPR Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')

    val_metric_cols = [col for col in history_df.columns if col.startswith('val_')]
    for col in val_metric_cols:
        axes[1].plot(history_df['epoch'], history_df[col], label=col)
    axes[1].set_title('Validation Metrics')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(run_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def train_one_run(model_name: str, data_bundle: dict, config: dict, seed: int) -> tuple[dict, pd.DataFrame]:
    run_config = deepcopy(config)
    run_config['seed'] = seed
    run_config['model_name'] = model_name

    set_seed(seed)
    device = torch.device(run_config['device'])
    model = build_model(model_name, data_bundle, run_config).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=run_config['lr'],
        weight_decay=run_config['weight_decay'],
    )

    train_edges = data_bundle['train_edges']
    edge_index = data_bundle['edge_index_train'].to(device)
    stopper = EarlyStopping(run_config['patience'])
    best_state = None
    history = []

    for epoch in tqdm(range(1, run_config['epochs'] + 1), desc=f'{model_name}-seed{seed}'):
        model.train()
        permutation = torch.randperm(train_edges.size(0))
        total_loss = 0.0

        for start in range(0, train_edges.size(0), run_config['batch_size']):
            batch_idx = permutation[start:start + run_config['batch_size']]
            batch = train_edges[batch_idx]
            users = batch[:, 0].to(device)
            pos_items = batch[:, 1].to(device)
            neg_items = sample_negative_items(
                users=users,
                train_user_pos_items=data_bundle['train_user_pos_items'],
                num_items=data_bundle['num_items'],
                device=device,
            )

            optimizer.zero_grad()
            pos_scores, neg_scores, reg_term = model.bpr_forward(users, pos_items, neg_items, edge_index)
            loss = bpr_loss(pos_scores, neg_scores, reg_term, run_config['reg_weight'])
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch.size(0)

        train_loss = total_loss / train_edges.size(0)
        val_metrics = evaluate_model(model, data_bundle, run_config, split='val')

        epoch_row = {'epoch': epoch, 'train_loss': train_loss}
        epoch_row.update({f'val_{key}': value for key, value in val_metrics.items()})
        history.append(epoch_row)

        improved = stopper.step(val_metrics[run_config['monitor']], epoch)
        if improved:
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        if stopper.should_stop:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    best_val_metrics = evaluate_model(model, data_bundle, run_config, split='val')
    test_metrics = evaluate_model(model, data_bundle, run_config, split='test')

    run_metrics = {
        'model_name': model_name,
        'seed': seed,
        'best_epoch': stopper.best_epoch,
        'monitor': run_config['monitor'],
        'best_monitor_value': stopper.best_value,
    }
    run_metrics.update({f'val_{key}': value for key, value in best_val_metrics.items()})
    run_metrics.update({f'test_{key}': value for key, value in test_metrics.items()})

    history_df = pd.DataFrame(history)
    run_dir = Path(run_config['artifacts_root']) / model_name / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)

    history_df.to_csv(run_dir / 'history.csv', index=False)
    with open(run_dir / 'metrics.json', 'w', encoding='utf-8') as fp:
        json.dump(run_metrics, fp, indent=2)

    serializable_config = deepcopy(run_config)
    serializable_config['top_k'] = list(serializable_config['top_k'])
    with open(run_dir / 'config.json', 'w', encoding='utf-8') as fp:
        json.dump(serializable_config, fp, indent=2)

    if run_config.get('save_curves', False):
        plot_history(history_df, run_dir)

    return run_metrics, history_df


def summarize_results(results: list[dict]) -> pd.DataFrame:
    results_df = pd.DataFrame(results)
    metric_cols = [col for col in results_df.columns if col.startswith('val_') or col.startswith('test_')]
    summary = results_df.groupby('model_name')[metric_cols].agg(['mean', 'std'])
    summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
    return summary.reset_index()


def run_all_baselines(config: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    data_bundle = load_movielens_implicit(config)
    all_results = []

    for model_name in ['bprmf', 'lightgcn']:
        for seed in config['seeds']:
            run_metrics, _ = train_one_run(model_name, data_bundle, config, seed)
            all_results.append(run_metrics)

    results_df = pd.DataFrame(all_results)
    summary_df = summarize_results(all_results)

    artifacts_root = Path(config['artifacts_root'])
    artifacts_root.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(artifacts_root / 'all_run_metrics.csv', index=False)
    summary_df.to_csv(artifacts_root / 'summary_metrics.csv', index=False)

    return results_df, summary_df

## Run Baselines

In [9]:
RUN_CONFIG = deepcopy(BASE_CONFIG)
# RUN_CONFIG['epochs'] = 5  # Mo comment nay neu can smoke test nhanh.
# RUN_CONFIG['seeds'] = [42]  # Mo comment nay neu can chay mot seed.

results_df, summary_df = run_all_baselines(RUN_CONFIG)
results_df

bprmf-seed42:   0%|          | 0/50 [00:00<?, ?it/s]W0712 14:22:51.797000 4168 Lib\site-packages\torch\autograd\profiler.py:858] Unexpected error importing torch.profiler._cupti_monitor
W0712 14:22:51.797000 4168 Lib\site-packages\torch\autograd\profiler.py:858] Traceback (most recent call last):
W0712 14:22:51.797000 4168 Lib\site-packages\torch\autograd\profiler.py:858]   File "d:\Code\Graph_ML\.venv\Lib\site-packages\torch\autograd\profiler.py", line 852, in _maybe_cupti_monitor
W0712 14:22:51.797000 4168 Lib\site-packages\torch\autograd\profiler.py:858]     )
W0712 14:22:51.797000 4168 Lib\site-packages\torch\autograd\profiler.py:858]       
W0712 14:22:51.797000 4168 Lib\site-packages\torch\autograd\profiler.py:858] ImportError: cannot import name '_cupti_monitor' from 'torch.profiler' (d:\Code\Graph_ML\.venv\Lib\site-packages\torch\profiler\__init__.py)
lightgcn-seed42:   6%|▌         | 3/50 [12:03<3:09:00, 241.29s/it]


KeyboardInterrupt: 

In [ ]:
summary_df